# Day 18 Revision Summary — Feature Engineering

- **Feature engineering is the biggest lever you have.** A feature is one input column; feature engineering means creating useful new columns, encoding text into numbers, and cleaning/scaling what's already there. A model can't invent information you didn't give it, but a good feature can hand it the answer directly.
- **The star demo: 0.70 → 0.99 with one column.** On a "healthy if x²+y² < 4" dataset, a logistic model on raw `[x, y]` scores ~0.695 (a straight line can't draw a circle). Adding one engineered column, `dist2 = x**2 + y**2`, lets the exact same model hit ~0.995 — one threshold on `dist2` splits the data cleanly.
- **One-hot encode categorical text, never label-encode it.** A column like `city = "KTM"` is text a model can't do math on. One-hot makes one 0/1 column per category (`is_KTM`, `is_PKR`, ...); mapping categories to 1, 2, 3 invents a fake ordering the model would wrongly learn from.
- **Scale numerics, impute gaps — inside a Pipeline.** Distance-based models (logistic, KNN, k-means) need `StandardScaler`; trees and forests don't. Fill numeric gaps with the median and categorical gaps with the most frequent value, and fit all of this on the *training* data only (inside a `Pipeline`) so no test information leaks in.
- **`ColumnTransformer` preps every column type in one object.** It routes numeric columns to `StandardScaler` and categorical columns to `OneHotEncoder(handle_unknown="ignore")` in a single step, wrapped in a `Pipeline` with the model — no manual steps to forget, and it evaluates honestly with `cross_val_score`.
- **Engineer generously, then select ruthlessly.** Useless columns add noise, hurt accuracy, and slow training down. `SelectKBest` (or a trained model's `feature_importances_`) tells you which columns actually earn their place.


## Homework (from the end of today's slide deck — Week 4 · Day 3)

Today's Drive folder has only a `slides` subfolder for Day 18 — no separate `homework` or `classwork` folders — so all of the assignment lives on the last two slides of `week4_day3_feature_engineering.html`:

1. Take any dataset and engineer one ratio or difference feature. Prove it helps with `cross_val_score`.
2. One-hot a categorical column two ways (`get_dummies` & `OneHotEncoder`) and confirm they match.
3. Read `formulas.md` in today's folder — one-hot, scaling, imputation, first principles.
4. Commit: `git add . && git commit -m "day 18"`.


## Assignment 1 — Engineer a ratio/difference feature and prove it helps (~task 1)

**What's being asked:** Pick any dataset, create one new column that is a *ratio* or a *difference* of two existing columns (the "everyday moves" from class: price ÷ area, end − start, etc.), and use `cross_val_score` to show the engineered feature actually raises the model's score versus the raw columns alone — the same honesty check used for the `dist2` demo in class (0.70 → 0.99).

**Approach:**
1. Load a small dataset with at least two numeric columns and a label (the sklearn `load_diabetes`/`load_wine` datasets or the class's own `x, y, healthy` circle data both work).
2. Build a baseline pipeline (`StandardScaler` + `LogisticRegression`, or `StandardScaler` + a regressor if the label is continuous) and score it on the *raw* columns with `cross_val_score(..., cv=5).mean()`.
3. Engineer one new column — a ratio (`col_a / col_b`) or a difference (`col_a - col_b`) — using `DataFrame.assign`.
4. Re-run the exact same pipeline and `cross_val_score` on raw + the new column.
5. Compare the two mean scores and write one sentence explaining why the new column helped (or didn't).


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# TODO 1: Build or load a small dataset with a label and at least two numeric
# columns. Reusing the class's own circle dataset is fine:
# rng = np.random.default_rng(42)
# x = rng.uniform(-3, 3, 600)
# y = rng.uniform(-3, 3, 600)
# healthy = ((x**2 + y**2) < 4).astype(int)
# df = pd.DataFrame({"x": x, "y": y})

# TODO 2: Baseline pipeline + score on the RAW columns only.
# model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
# raw = df[["x", "y"]]
# raw_score = cross_val_score(model, raw, healthy, cv=5).mean()
# print("raw:", raw_score)

# TODO 3: Engineer ONE ratio or difference column with df.assign(...).
# eng = raw.assign(new_col=...)

# TODO 4: Score the SAME pipeline on raw + the new column.
# eng_score = cross_val_score(model, eng, healthy, cv=5).mean()
# print("engineered:", eng_score)

# TODO 5: One sentence — why did (or didn't) the new column help?


## Assignment 2 — One-hot encode a categorical column two ways (~task 2)

**What's being asked:** Take any categorical column and one-hot encode it with both `pd.get_dummies` (the quick, exploratory way) and `sklearn.preprocessing.OneHotEncoder` (the production way that survives unseen categories at prediction time), then confirm the two results actually agree.

**Approach:**
1. Build a small `DataFrame` with one text column (the class's `city = ["KTM", "PKR", "BRT", "KTM"]` example, or your own categories).
2. Encode it with `pd.get_dummies(df["col"], prefix="col")`.
3. Encode it again with `OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit_transform(df[["col"]])`, and check `.categories_` for the column order sklearn picked.
4. Sort both results' columns into the same order and compare the underlying values (e.g. with `.to_numpy()` and `np.array_equal`, or a boolean `.equals()` after aligning columns) to confirm they match.
5. Note when you'd reach for each: `get_dummies` for a quick notebook look, `OneHotEncoder` inside a real `Pipeline`/`ColumnTransformer` because it survives categories it never saw in training.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

# TODO 1: A tiny table with one text column, "city".
# df = pd.DataFrame({"city": ["KTM", "PKR", "BRT", "KTM"]})

# TODO 2: Quick way — pd.get_dummies.
# dummies = pd.get_dummies(df["city"], prefix="city")
# print(dummies)

# TODO 3: Production way — OneHotEncoder.
# enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
# encoded = enc.fit_transform(df[["city"]])
# print(enc.categories_)
# print(encoded)

# TODO 4: Align column order and confirm the two encodings match
# (dummies columns are alphabetical by category, same as enc.categories_ here).
# dummies_sorted = dummies.sort_index(axis=1)
# print(np.array_equal(dummies_sorted.to_numpy().astype(int), encoded.astype(int)))


## Assignment 3 — Read `formulas.md` (~task 3)

**What's being asked:** No coding here — read `formulas.md` referenced in today's Drive folder, covering one-hot encoding, scaling, and imputation from first principles (e.g. the `z = (x - mean) / std` formula behind `StandardScaler`). Use it as a written reference alongside Assignments 1 and 2 above.
